# LAB 01 — Từ xử lý văn bản đến tìm kiếm

Notebook chạy thí nghiệm trên shard C4 được cung cấp, gồm 30.000 record. Mã văn bản là số thứ tự dòng bắt đầu từ 1 (`D00001`, ...). Ba pipeline dùng chung corpus, query đánh giá và nhãn thủ công. Nhãn này không phải nhãn do giảng viên cung cấp.

**Các giả định do đề chưa quy định cụ thể**

- Pipeline A: chuyển chữ thường, tách theo khoảng trắng; dấu câu vẫn dính với token.
- Pipeline B: thay dấu câu/ký tự điều khiển Unicode bằng khoảng trắng, tách theo khoảng trắng rồi bỏ stopword tiếng Anh có sẵn trong `sklearn`.
- Pipeline C: chuẩn hóa như B rồi dùng WordPiece được train cục bộ trên corpus (`vocab_size=20,000`, `min_frequency=2`); không tải model hay dùng mạng.
- OOV là tỷ lệ token của cùng tập query đánh giá không có trong vocabulary đã fit của pipeline đó.
- Nhãn relevance là tập nhỏ được gán thủ công sau khi xem nội dung record; đây không phải đánh giá đầy đủ mọi văn bản.
- Thí nghiệm 1 dùng token Pipeline B đã chuẩn hóa để giảm ảnh hưởng của các biến thể dấu câu khi xem term.


## 1. Nạp thư viện

In [1]:
from pathlib import Path
import gc
import math
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from implementation import (
    build_vocabulary, compute_counts, compute_tf, compute_idf, compute_tfidf,
    cosine_similarity, load_corpus, tokenize_minimal, tokenize_normalized, normalize_punctuation,
    train_subword_tokenizer, tokenize_subword, fit_sparse_index, search_index,
    query_oov_rate, evaluate_pipeline, write_results_csv,
)

DATA_PATH = Path("c4-train.00000-of-01024-30K.json.gz")
OUTPUT_PATH = Path("results.csv")


## 2. Nạp dữ liệu

In [2]:
records = load_corpus(DATA_PATH)
texts = [record["text"] for record in records]
document_ids = [record["document_id"] for record in records]
assert len(records) > 0


## 3. Kiểm tra dữ liệu ban đầu

In [3]:
print(f"Số văn bản đã nạp: {len(records):,}")
print("Mã đầu tiên:", document_ids[0], "mã cuối:", document_ids[-1])
print("Các trường của record:", sorted(records[0]))
print("URL văn bản đầu:", records[0]["url"])
print("Trích đoạn văn bản đầu:", records[0]["text"][:300].replace("\n", " "))


Số văn bản đã nạp: 30,000
Mã đầu tiên: D00001 mã cuối: D30000
Các trường của record: ['document_id', 'text', 'timestamp', 'url']
URL văn bản đầu: https://klyq.com/beginners-bbq-class-taking-place-in-missoula/
Trích đoạn văn bản đầu: Beginners BBQ Class Taking Place in Missoula! Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class fo


## 4. Thí nghiệm 1 — Biểu diễn sparse (Pipeline B)

In [4]:
tokens_B = [tokenize_normalized(text) for text in texts]
index_B = fit_sparse_index(tokens_B)
del tokens_B
gc.collect()
N, V = index_B["tfidf"].shape
nnz = int(index_B["tfidf"].nnz)
sparsity = 1.0 - nnz / (N * V)
print({"N": N, "V": V, "kích_thước_ma_trận": index_B["tfidf"].shape,
       "nnz": nnz, "sparsity": sparsity})


{'N': 30000, 'V': 197022, 'kích_thước_ma_trận': (30000, 197022), 'nnz': 3756121, 'sparsity': 0.9993645174988919}


## 5. Từ vựng, số văn bản chứa từ (DF), IDF và TF-IDF của một văn bản

In [5]:
terms = index_B["terms"]
df = index_B["df"]
idf = index_B["idf"]
top_df_indices = np.lexsort((terms, -df))[:20]
top_idf_indices = np.lexsort((terms, -idf))[:20]
print("20 từ có DF cao nhất (từ, DF):")
for i in top_df_indices:
    print(str(terms[i]), int(df[i]))
print("\n20 từ có IDF cao nhất (từ, IDF):")
for i in top_idf_indices:
    print(str(terms[i]), float(idf[i]))

selected_index = 0
selected_id = document_ids[selected_index]
row = index_B["tfidf"].getrow(selected_index)
ranked_terms = sorted(zip(row.indices, row.data), key=lambda item: (-item[1], str(terms[item[0]])))[:20]
print(f"\nVăn bản được chọn: {selected_id}")
print("Trích đoạn:", texts[selected_index][:300].replace("\n", " "))
print("Các từ có TF-IDF cao nhất (từ, TF-IDF):")
for term_index, value in ranked_terms:
    print(str(terms[term_index]), float(value))


20 từ có DF cao nhất (từ, DF):
s 16885
time 9540
new 8926
t 8757
like 7977
just 7505
make 7289
use 6369
work 6304
need 5980
way 5811
best 5744
people 5735
years 5525
help 5425
good 5421
1 5382
year 5372
2 5272
great 5184

20 từ có IDF cao nhất (từ, IDF):
$$p 10.308952660644293
$1000+ 10.308952660644293
$100000 10.308952660644293
$100080000 10.308952660644293
$1007 10.308952660644293
$100aus 10.308952660644293
$100k+ 10.308952660644293
$100ks 10.308952660644293
$100s 10.308952660644293
$101 10.308952660644293
$1024 10.308952660644293
$1035 10.308952660644293
$1050 10.308952660644293
$10705 10.308952660644293
$1095 10.308952660644293
$1100 10.308952660644293
$110000112000 10.308952660644293
$114 10.308952660644293
$1157 10.308952660644293
$115k 10.308952660644293

Văn bản được chọn: D00001
Trích đoạn: Beginners BBQ Class Taking Place in Missoula! Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join 

## Phần cài đặt cốt lõi — Đối chiếu với cách tính tham chiếu

Ở đây scikit-learn chỉ tạo ma trận đếm cho corpus nhỏ. TF và IDF được tính trực tiếp theo công thức của đề, không dùng `TfidfVectorizer`.

In [6]:
core_documents = ["cat eats fish", "dog eats fish", "cat likes fish"]
core_vocabulary = build_vocabulary(core_documents)
student_counts = compute_counts(core_documents, core_vocabulary)
student_tf = compute_tf(student_counts)
student_idf = compute_idf(student_counts)
student_tfidf = compute_tfidf(student_tf, student_idf)

reference_vectorizer = CountVectorizer()
reference_counts = reference_vectorizer.fit_transform(core_documents).toarray()
reference_terms = reference_vectorizer.get_feature_names_out()
assert list(reference_terms) == list(core_vocabulary)
reference_tf = reference_counts / reference_counts.sum(axis=1, keepdims=True)
reference_df = np.count_nonzero(reference_counts, axis=0)
reference_idf = np.log(len(core_documents) / reference_df)
reference_tfidf = reference_tf * reference_idf
np.testing.assert_allclose(student_counts, reference_counts, atol=1e-9)
np.testing.assert_allclose(student_tf, reference_tf, atol=1e-9)
np.testing.assert_allclose(student_idf, reference_idf, atol=1e-9)
np.testing.assert_allclose(student_tfidf, reference_tfidf, atol=1e-9)
print("Implementation của sinh viên khớp với ma trận đếm tham chiếu và công thức trực tiếp của đề (sai số 1e-9).")
print("IDF tham chiếu:", dict(zip(reference_terms, reference_idf)))


Implementation của sinh viên khớp với ma trận đếm tham chiếu và công thức trực tiếp của đề (sai số 1e-9).
IDF tham chiếu: {'cat': np.float64(0.4054651081081644), 'dog': np.float64(1.0986122886681098), 'eats': np.float64(0.4054651081081644), 'fish': np.float64(0.0), 'likes': np.float64(1.0986122886681098)}


## 6. Pipeline A — Tối giản

In [7]:
tokens_A = [tokenize_minimal(text) for text in texts]
index_A = fit_sparse_index(tokens_A)
del tokens_A
gc.collect()
print("Index của Pipeline A:", index_A["tfidf"].shape,
      "nnz=", index_A["tfidf"].nnz,
      "số token trung bình/văn bản=", index_A["average_tokens"],
      "sparsity=", index_A["sparsity"])


Index của Pipeline A: (30000, 473388) nnz= 5519587 số token trung bình/văn bản= 361.08906666666667 sparsity= 0.9996113415774516


## 7. Pipeline B — Chuẩn hóa

In [8]:
print("Index của Pipeline B:", index_B["tfidf"].shape,
      "nnz=", index_B["tfidf"].nnz,
      "số token trung bình/văn bản=", index_B["average_tokens"],
      "sparsity=", index_B["sparsity"])


Index của Pipeline B: (30000, 197022) nnz= 3756121 số token trung bình/văn bản= 197.59403333333333 sparsity= 0.9993645174988919


## 8. Pipeline C — Subword mở rộng

In [9]:
# WordPiece được train cục bộ; cấu hình và corpus cố định để kết quả có thể lặp lại.
subword_tokenizer = train_subword_tokenizer(texts, vocab_size=20_000)
tokenize_C = lambda text: tokenize_subword(text, subword_tokenizer)
tokens_C = [tokenize_C(text) for text in texts]
index_C = fit_sparse_index(tokens_C)
del tokens_C
gc.collect()
print("Kích thước vocabulary tokenizer Pipeline C:", subword_tokenizer.get_vocab_size())
print("Index của Pipeline C:", index_C["tfidf"].shape,
      "nnz=", index_C["tfidf"].nnz,
      "số token trung bình/văn bản=", index_C["average_tokens"],
      "sparsity=", index_C["sparsity"])


Kích thước vocabulary tokenizer Pipeline C: 20000
Index của Pipeline C: (30000, 18615) nnz= 5739474 số token trung bình/văn bản= 417.31216666666666 sparsity= 0.9897224926134838


## 9. Các metric so sánh

In [10]:
pipeline_indices = {"A": (index_A, tokenize_minimal),
                    "B": (index_B, tokenize_normalized),
                    "C": (index_C, tokenize_C)}


## 10. Search — Các query gợi ý trong đề

In [11]:
suggested_queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
]
for query in suggested_queries:
    print(f"\nTRUY VẤN: {query}")
    for result in search_index(query, 5, index_B, tokenize_normalized, records):
        print("Hạng", result["rank"], "Mã văn bản", result["document_id"],
              f"điểm={result['similarity']:.4f}", result["preview"])



TRUY VẤN: medical image classification
Hạng 1 Mã văn bản D18972 điểm=0.4465 The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in an environmentally responsible manner. The environmental classification system was developed for Finland, and it takes into accou
Hạng 2 Mã văn bản D08528 điểm=0.3678 History of maize classification. How races used in classification. Geographical distribution. Existing races of maize in Mexico.
Hạng 3 Mã văn bản D19909 điểm=0.2356 Download League Of Legends Wallpapers in high-quality for your desktop and smart-phone in wide-screen and HD resolution. Right click on image, and select “Save Image as League Of Legends Wallpapers” to download image to your desktop, laptop. If you are browsing website by mobile 
Hạng 4 Mã văn bản D15683 điểm=0.2321 - Group Image: Provided functionality of group's image, user can now view and change the group's image also. - Profile im

## 11. Tập đánh giá

Đây là **tập đánh giá thủ công cho thí nghiệm**, không phải tập do giảng viên cung cấp. Mã văn bản liên quan được gán sau khi xem record tương ứng. Cả ba pipeline dùng cùng nhãn.

In [12]:
evaluation_queries = [
    {"query_id": "e1", "query": "RTS environmental classification construction project"},
    {"query_id": "e2", "query": "medical second opinion advice treatment"},
    {"query_id": "e3", "query": "big data healthcare systems"},
    {"query_id": "e4", "query": "foreign language instructor"},
    {"query_id": "e5", "query": "New Jersey State barbecue championship"},
]
relevant_by_query = {
    "e1": {"D18972"},
    "e2": {"D08371", "D10891"},
    "e3": {"D11120"},
    "e4": {"D04076", "D00702"},
    "e5": {"D13763"},
}
assert set(relevant_by_query) == {q["query_id"] for q in evaluation_queries}
print("Số query đánh giá:", len(evaluation_queries))
print("Nhãn liên quan:", {key: sorted(value) for key, value in relevant_by_query.items()})


Số query đánh giá: 5
Nhãn liên quan: {'e1': ['D18972'], 'e2': ['D08371', 'D10891'], 'e3': ['D11120'], 'e4': ['D00702', 'D04076'], 'e5': ['D13763']}


## 12. Precision@5

In [13]:
result_rows_by_pipeline = {}
aggregate_by_pipeline = {}
for name, (index, token_fn) in pipeline_indices.items():
    rows, aggregate = evaluate_pipeline(
        name, index, token_fn, records, evaluation_queries, relevant_by_query, top_k=5
    )
    result_rows_by_pipeline[name] = rows
    aggregate_by_pipeline[name] = aggregate
    print(f"{name}: P@5 = {aggregate['precision_at_5']:.4f}")


A: P@5 = 0.2800


B: P@5 = 0.2800


C: P@5 = 0.2800


## 13. Recall@5

In [14]:
for name, aggregate in aggregate_by_pipeline.items():
    print(f"{name}: Recall@5 = {aggregate['recall_at_5']:.4f}")


A: Recall@5 = 1.0000
B: Recall@5 = 1.0000
C: Recall@5 = 1.0000


## 14. MRR

In [15]:
for name, aggregate in aggregate_by_pipeline.items():
    print(f"{name}: MRR = {aggregate['mrr']:.4f}")


A: MRR = 0.9000
B: MRR = 0.9000
C: MRR = 1.0000


## 15. Kết quả tìm kiếm và dữ liệu đối chiếu

In [16]:
# In 5 kết quả xếp hạng cho từng query đánh giá ở mỗi pipeline.
for name, (index, token_fn) in pipeline_indices.items():
    print(f"\nQUY TRÌNH {name}")
    for item in evaluation_queries:
        print(f"\n{item['query_id']}: {item['query']}")
        for result in search_index(item["query"], 5, index, token_fn, records):
            relevant = result["document_id"] in relevant_by_query[item["query_id"]]
            print(result["rank"], result["document_id"],
                  f"điểm={result['similarity']:.4f}", f"liên quan={relevant}", result["preview"])



QUY TRÌNH A

e1: RTS environmental classification construction project
1 D18972 điểm=0.6356 liên quan=True The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in an environmentally responsible manner. The environmental classification system was developed for Finland, and it takes into accou
2 D07361 điểm=0.2252 liên quan=False CONSTRUCTION EQUIPMENTS 8 Small Construction . CONSTRUCTION EQUIPMENTS We are regularly supplying New brand Construction Equipments to African & Middle East countries. We supply these equipments from renowned manufacturers from India. All our manufacturers ... list of other Small
3 D18135 điểm=0.2056 liên quan=False 21, Adeboye Solanke Avenue, off Allen Avenue Ikeja, Lagos State, Nigeria. The Institute of Environmental Accountants is a specialized accounting institution for the good purpose of promoting professionalism in environmental quality. We deals with the ways 

1 D02965 điểm=0.2844 liên quan=False Although New Jersey is the fourth smallest state in the United States, it is the 11th most populous state in the country with over an estimated 8,900,000 people residing there. It is also statistically deemed the second wealthiest. New Jersey is bordered north and east by New Yor
2 D13763 điểm=0.2442 liên quan=True THE NEW JERSEY STATE BARBECUE CHAMPIONSHIP ( njbbq.com) will be held July 10 to 12 in North Wildwood. The 17th annual throwdown, sanctioned by the top-dog Kansas City Barbeque Society, brings 72 teams from around the country to compete in four categories—chicken, pork ribs, pulle
3 D13112 điểm=0.2198 liên quan=False Wholesale Nfl Jerseys China | Cheap Jerseys Everyday Wholesale, Free Worldwide Shipping! baseball with their World Series victory over the Tampa Bay Rays. They has a large and loyal fan base wholesale nfl jerseys China even though it has been one wholesale nfl jerseys Final Sales
4 D21572 điểm=0.1940 liên quan=False Just as th

1 D04076 điểm=0.4594 liên quan=True Looking for Spanish language instructor to improve my reading, writing and speaking skills. We require the tutor to travel to our location at Gwal Pahari. A certification or degree in Spanish Language is preferred. The Spanish language instructor should be punctual and be willing
2 D00702 điểm=0.4047 liên quan=True Program in Teaching French as a Foreign Language was established in 1985 under the Department of Foreign Education. 2 full professors,3 assistant professors and 3 research assistant are currently working in the department. The Program in Teaching French as a Foreign Language offe
3 D25429 điểm=0.3001 liên quan=False Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses whatever language your phone is set up to use, so to change it you have to pick a different language for your entire phone. Do this through the Settings app via
4 D00391 điểm=0.2843 liên quan=False I’ve never 

1 D13763 điểm=0.3873 liên quan=True THE NEW JERSEY STATE BARBECUE CHAMPIONSHIP ( njbbq.com) will be held July 10 to 12 in North Wildwood. The 17th annual throwdown, sanctioned by the top-dog Kansas City Barbeque Society, brings 72 teams from around the country to compete in four categories—chicken, pork ribs, pulle
2 D25249 điểm=0.3271 liên quan=False Denver has the most saturated market of roofers in the country. Choose quality over quantity. Avoid these 5 mistakes when choosing a Roofer! Take a look at some of the pictures from our Company's Summer Barbecue!
3 D02965 điểm=0.2994 liên quan=False Although New Jersey is the fourth smallest state in the United States, it is the 11th most populous state in the country with over an estimated 8,900,000 people residing there. It is also statistically deemed the second wealthiest. New Jersey is bordered north and east by New Yor
4 D21147 điểm=0.2840 liên quan=False The teacher put the students’ drawings on the wall. I drink nothing but apple 

## 16. Lưu results.csv

In [17]:
all_result_rows = [row for name in pipeline_indices for row in result_rows_by_pipeline[name]]
write_results_csv(all_result_rows, OUTPUT_PATH)
summary = pd.DataFrame([aggregate_by_pipeline[name] for name in pipeline_indices]).rename(columns={
    "pipeline": "Pipeline", "vocabulary_size": "Cỡ từ vựng",
    "average_tokens_per_document": "Token TB/văn bản", "matrix_sparsity": "Sparsity",
    "oov_rate": "Tỷ lệ OOV", "precision_at_5": "P@5",
    "recall_at_5": "Recall@5", "mrr": "MRR"})
print(summary.to_string(index=False))
print(f"\nĐã lưu {len(all_result_rows)} dòng kết quả xếp hạng vào {OUTPUT_PATH.resolve()}")


Pipeline  Cỡ từ vựng  Token TB/văn bản  Sparsity  Tỷ lệ OOV  P@5  Recall@5  MRR
       A      473388        361.089067  0.999611        0.0 0.28       1.0  0.9
       B      197022        197.594033  0.999365        0.0 0.28       1.0  0.9
       C       18615        417.312167  0.989722        0.0 0.28       1.0  1.0

Đã lưu 75 dòng kết quả xếp hạng vào D:\Study\Semester 7\NLP\Lab01\results.csv


## 17. Ablation phụ để trả lời câu hỏi preprocessing

So sánh A/B ban đầu thay đổi cả xử lý dấu câu lẫn stopword nên không tách riêng được từng tác động. Cell dưới đo ảnh hưởng của chuyển chữ thường trên cùng tokenizer; đồng thời đo chuẩn hóa dấu câu khi chưa bỏ stopword và so sánh thêm với bản có bỏ stopword.

In [18]:
def tokenize_normalized_no_stopwords(text):
    return normalize_punctuation(text).split()

# Tách riêng tác động chữ hoa/thường: giữ nguyên tokenizer khoảng trắng.
case_sensitive_vectorizer = CountVectorizer(
    analyzer=lambda document: document.split(), lowercase=False
)
case_sensitive_vectorizer.fit(texts)
case_sensitive_vocab_size = len(case_sensitive_vectorizer.vocabulary_)
lowercase_vocab_size = int(index_A["tfidf"].shape[1])

# Đo chuẩn hóa dấu câu riêng, chưa áp dụng bước bỏ stopword.
tokens_B_no_stop = [tokenize_normalized_no_stopwords(text) for text in texts]
index_B_no_stop = fit_sparse_index(tokens_B_no_stop)
del tokens_B_no_stop
gc.collect()
_, no_stop_aggregate = evaluate_pipeline(
    "B_no_stopwords", index_B_no_stop, tokenize_normalized_no_stopwords,
    records, evaluation_queries, relevant_by_query, top_k=5
)
print("Vocabulary phân biệt hoa-thường:", case_sensitive_vocab_size)
print("Vocabulary sau chuyển chữ thường (Pipeline A):", lowercase_vocab_size)
print("Số mục được gộp sau khi chuyển chữ thường:",
      case_sensitive_vocab_size - lowercase_vocab_size)
print("Chuẩn hóa nhưng giữ stopword:", {
    "Cỡ từ vựng": index_B_no_stop["tfidf"].shape[1],
    "Token TB/văn bản": index_B_no_stop["average_tokens"],
    "Sparsity": index_B_no_stop["sparsity"],
    "P@5": no_stop_aggregate["precision_at_5"],
    "Recall@5": no_stop_aggregate["recall_at_5"],
    "MRR": no_stop_aggregate["mrr"],
})
print("Ví dụ dấu câu, trước và sau chuẩn hóa:", {
    "t-shirt": (tokenize_minimal("t-shirt"), tokenize_normalized_no_stopwords("t-shirt")),
    "can't": (tokenize_minimal("can't"), tokenize_normalized_no_stopwords("can't")),
})


Vocabulary phân biệt hoa-thường: 542753
Vocabulary sau chuyển chữ thường (Pipeline A): 473388
Số mục được gộp sau khi chuyển chữ thường: 69365
Chuẩn hóa nhưng giữ stopword: {'Cỡ từ vựng': 197338, 'Token TB/văn bản': 369.8599, 'Sparsity': 0.9991370089220863, 'P@5': 0.28, 'Recall@5': 1.0, 'MRR': 0.9}
Ví dụ dấu câu, trước và sau chuẩn hóa: {'t-shirt': (['t-shirt'], ['t', 'shirt']), "can't": (["can't"], ['can', 't'])}


## 18. Trả lời câu hỏi thực nghiệm

### Phần D — Biểu diễn sparse và trọng số

Mỗi vị trí trong vector ứng với một từ trong vocabulary chung. Vì vậy mọi văn bản có cùng chiều V, và văn bản/query có thể so sánh trên cùng hệ tọa độ. Phần lớn vị trí bằng 0 vì một văn bản chỉ dùng một phần nhỏ vocabulary.

Từ có DF cao không nhất thiết có TF-IDF cao: DF tăng thì IDF giảm. Từ có IDF cao cũng không có TF-IDF cao trong mọi văn bản; nếu từ không xuất hiện thì TF và TF-IDF của nó bằng 0. Xem các bảng DF/IDF và TF-IDF đã tính ở trên.

### Phần F — So sánh preprocessing

- **Chuyển chữ thường:** cùng tokenizer tách khoảng trắng cho vocabulary phân biệt hoa-thường 542.753 mục; sau khi chuyển chữ thường còn 473.388, giảm 69.365 mục do các biến thể nhập làm một.
- **Bỏ stopword:** bản chuẩn hóa nhưng giữ stopword có V=197.338 và trung bình 369,86 token/văn bản. Pipeline B có V=197.022 và trung bình 197,59 token/văn bản. Trên 5 query có nhãn, cả hai có P@5=0,28, Recall@5=1,0, MRR=0,90. Kết quả nhỏ này không chứng minh bỏ stopword luôn có lợi hay có hại.
- **Dấu câu:** khi chưa bỏ stopword, chuẩn hóa dấu câu cho V=197.338 và trung bình 369,86 token/văn bản. Ví dụ `t-shirt` thành `t`, `shirt`; `can't` thành `can`, `t`. Ranh giới từ thay đổi và đôi khi còn lại mảnh từ.
- **Pipeline có sparsity cao nhất:** A theo tỷ lệ đo được. Sparsity phụ thuộc cả số chiều vocabulary lẫn số phần tử khác 0.
- **Search tốt nhất trong tập này:** P@5 và Recall@5 bằng nhau ở A/B/C. C có MRR cao nhất trên tập nhãn thủ công. Kết luận chỉ áp dụng cho 5 query này.
- **Vocabulary và chất lượng search:** vocabulary nhỏ hơn không tự động làm search tốt hơn. Trong lần chạy này C có vocabulary nhỏ nhất và MRR cao hơn, còn P@5/Recall@5 bằng nhau. Năm query chưa đủ để kết luận quan hệ tổng quát.

### Phần G–H — Search và đánh giá

Bốn query gợi ý trong đề là ví dụ tìm kiếm, không có nhãn relevance của giảng viên. Kết quả thật của Pipeline B được in ở Phần 10. Cả ba pipeline được đánh giá bằng cùng năm query và cùng mã văn bản liên quan. P@5, Recall@5 và MRR được tính ở các cell phía trên.

### Phần I — Ví dụ phân tích lỗi

- **Khớp tốt e1:** query về hệ thống phân loại môi trường RTS cho dự án xây dựng xếp `D18972` hạng 1; preview đúng chủ đề này.
- **Khớp tốt e2:** query về ý kiến y khoa thứ hai xếp `D08371` hạng 1; nội dung nói về dịch vụ xin ý kiến y khoa, tư vấn và điều trị.
- **Khớp từ nhưng sai chủ đề:** query `transformer language model` xếp `D27937` đầu ở Pipeline B; preview nói về transformer/bo mạch điện. Từ trùng không đồng nghĩa với đúng chủ đề NLP.
- **Khớp từ nhưng sai chủ đề:** query `natural language processing` xếp văn bản quy định an toàn thực phẩm `D08706` đầu.
- **Lỗi thứ hạng trong tập đánh giá:** với e5, A/B xếp văn bản về giải barbecue `D13763` hạng 2 sau `D02965`, còn C xếp `D13763` hạng 1. Xem đầy đủ thứ hạng trong `results.csv` và Phần 15.

### Phần J — Giả thuyết cho representation tiếp theo

Representation dựa trên ngữ cảnh phân bố hoặc contextual embedding có thể đặt các cách diễn đạt gần nghĩa gần nhau dù chúng ít trùng từ. Thí nghiệm TF-IDF này chưa kiểm chứng giả thuyết đó; cần một thí nghiệm và bộ đánh giá riêng.
